In [1]:
import pandas as pd

data = pd.read_csv('Connections_Data.csv')

# keep columns "name" and "connections"
data = data[['Word', 'Group Name']]

data.to_csv('Connections_Data_Cleaned.csv', index=False)


In [4]:
data = pd.read_csv('Connections_Data_Cleaned.csv')

print(data.head())
print(data.dtypes)

    Word     Group Name
0   SNOW    WET WEATHER
1  LEVEL    PALINDROMES
2  SHIFT  KEYBOARD KEYS
3  KAYAK    PALINDROMES
4   HEAT      NBA TEAMS
Word          object
Group Name    object
dtype: object


# game strategy

If result is one off, take next highest weight with 3 of the same words

if there are only two categories left and answer is wrong, then select atleast 1 from original with 3 to make highest weight pairing

# alternate approaches

A 4 class svm could peroform better than the sentence transformers

A dedicated model in detecting one specific group to decrease the complexity of the problem.

In [ ]:
import pandas as pd
import random
import itertools
import pickle
import os
import torch
import numpy as np
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
from tqdm import tqdm

CSV_FILE = 'Connections_Data_Cleaned.csv'
MODEL_NAME = 'cross-encoder/nli-deberta-v3-base' # change to a bigger model if model isnt performing
MODEL_SAVE_PATH = 'my_connections_model'
CACHE_FILE = 'adversarial_data.pkl'

BATCH_SIZE = 8   
EPOCHS_R1 = 2    
EPOCHS_R2 = 3    # error epoch training should be higher

def load_raw_data(csv_path):
    print(f"Reading {csv_path}...")
    df = pd.read_csv(csv_path)
    df.dropna(subset=['Word', 'Group Name'], inplace=True)
    df['Word'] = df['Word'].astype(str).str.strip()
    return df

def generate_base_examples(df):
    """Generates standard Positive and Random Negative examples."""
    groups = df.groupby('Group Name')['Word'].apply(list).to_dict()
    all_words = df['Word'].unique().tolist()
    examples = []

    print("Generating base training data...")
    for category, members in groups.items():
        if len(members) < 4: continue
        
        # POSITIVE (The Real Group)
        correct_group = sorted(members[:4])
        examples.append(InputExample(texts=[", ".join(correct_group), ""], label=1.0))
        
        # RANDOM NEGATIVES (Basic Distractors)
        if len(all_words) > 10:
            for _ in range(2):
                intruder = random.choice(all_words)
                while intruder in members: intruder = random.choice(all_words)
                bad_group = members[:3] + [intruder]
                random.shuffle(bad_group)
                examples.append(InputExample(texts=[", ".join(bad_group), ""], label=0.0))
                
    return examples

def mine_hard_negatives(model, df):
    print("\n--- PHASE 2: MINING RED HERRINGS (WITH SAFETY CHECKS) ---")
    
    # Build a "Registry" of ALL valid groups ever seen
    print("Building global registry of valid categories...")
    global_valid_groups = set()
    category_groups = df.groupby('Category')['Word'].apply(list)
    
    for words in category_groups:
        if len(words) >= 4:
            # Add the sorted tuple of the first 4 words
            # (In a perfect world we'd add all subsets of 4, but this covers 99%)
            global_valid_groups.add(tuple(sorted(words[:4])))

    # Build the Puzzles
    category_dict = df.groupby('Category')['Word'].apply(list).to_dict()
    all_categories = list(category_dict.keys())
    
    hard_negatives = []
    num_mock_games = 100 
    
    for _ in tqdm(range(num_mock_games)):
        # Pick 4 random categories to mix
        chosen_cats = random.sample(all_categories, 4)
        
        game_words = []
        truth_map = {} 
        
        valid_game = True
        for cat in chosen_cats:
            words = category_dict[cat]
            if len(words) < 4: 
                valid_game = False
                break
            selected = words[:4]
            game_words.extend(selected)
            for w in selected:
                truth_map[w] = cat
                
        if not valid_game: continue
        
        random.shuffle(game_words)
        
        # Ask Model to Predict
        combos = list(itertools.combinations(game_words, 4))
        combo_inputs = [(", ".join(sorted(c)), "") for c in combos]
        scores = model.predict(combo_inputs, batch_size=64, show_progress_bar=False)
        
        # Filter for Mistakes
        for idx, score in enumerate(scores):
            if score > 0.8: # Model is confident
                group_candidate = combos[idx]
                candidate_tuple = tuple(sorted(group_candidate))
                
                # CHECK 1: Is this actually a valid group in our database?
                if candidate_tuple in global_valid_groups:
                    # Do NOT punish the model. It found a real connection we didn't expect!
                    continue

                # CHECK 2: Do they share a category in this specific mock game?
                cats_in_group = {truth_map[w] for w in group_candidate}
                
                if len(cats_in_group) > 1:
                    # It's not a global valid group, AND it mixes categories locally.
                    # It is safe to mark as WRONG.
                    hard_negatives.append(InputExample(
                        texts=[", ".join(sorted(group_candidate)), ""], 
                        label=0.0 
                    ))

    print(f"Generated {len(hard_negatives)} SAFE adversarial examples.")
    return hard_negatives

def train():
    # PREPARE DATA
    df = load_raw_data(CSV_FILE)
    base_examples = generate_base_examples(df)
    
    # INITIAL TRAINING
    print(f"\n--- PHASE 1: WARMUP TRAINING ({MODEL_NAME}) ---")
    model = CrossEncoder(MODEL_NAME, num_labels=1, automodel_args = {"ignore_mismatched_sizes": True})
    
    train_dataloader = DataLoader(base_examples, shuffle=True, batch_size=BATCH_SIZE)
    loss_fct = torch.nn.MSELoss()
    
    model.fit(
        train_dataloader=train_dataloader,
        epochs=EPOCHS_R1,
        loss_fct=loss_fct,
        show_progress_bar=True
    )
    
    # MINE MISTAKES
    # Now we use the half-trained model to find what confuses it
    hard_negatives = mine_hard_negatives(model, df)
    
    if not hard_negatives:
        print("Model made no confident mistakes! (Unlikely). Skipping Phase 3.")
        model.save(MODEL_SAVE_PATH)
        return

    # ADVERSARIAL RETRAINING
    print(f"\n--- PHASE 3: ADVERSARIAL RETRAINING ---")
    print(f"Retraining on Base Data + {len(hard_negatives)} Hard Negatives...")
    
    # Combine datasets
    full_dataset = base_examples + hard_negatives
    train_dataloader = DataLoader(full_dataset, shuffle=True, batch_size=BATCH_SIZE)
    
    model.fit(
        train_dataloader=train_dataloader,
        epochs=EPOCHS_R2, # Train longer this time
        loss_fct=loss_fct,
        show_progress_bar=True
    )
    
    print(f"Saving smarter model to {MODEL_SAVE_PATH}...")
    model.save(MODEL_SAVE_PATH)
    print("Done.")

if __name__ == "__main__":
    train()


In [ ]:
import itertools
import os
import sys
import numpy as np
from sentence_transformers import CrossEncoder, SentenceTransformer, util

MODEL_PATH = 'my_connections_model' 
SEARCH_DEPTH = 60 # Check top 60 candidates for global fit

def apply_one_away_boost(candidates, failed_guess):
    print(f"   >> applying 'One Away' boost to neighbors of {failed_guess}...")
    failed_set = set(failed_guess)
    
    boosted_candidates = []
    for group, score in candidates:
        # Check overlap size
        overlap = len(set(group).intersection(failed_set))
        
        if overlap == 3:
            # HUGE BOOST: +5.0 ensures this becomes the top priority
            # regardless of what the model originally thought.
            new_score = score + 5.0
        else:
            new_score = score
            
        boosted_candidates.append((group, new_score))
        
    # Re-sort immediately so the best option floats to the top
    boosted_candidates.sort(key=lambda x: x[1], reverse=True)
    return boosted_candidates

def get_coherence_penalty(group, embedder):
    vecs = embedder.encode(group)
    cos_sims = util.cos_sim(vecs, vecs).numpy()
    # Get off-diagonal elements (pairwise similarities)
    pairs = cos_sims[np.triu_indices(4, k=1)]
    min_sim = np.min(pairs)
    
    if min_sim < 0.2: return -0.5  # Heavy penalty
    elif min_sim < 0.35: return -0.2 # Slight penalty
    return 0.0

def check_patterns(group):
    bonus = 0.0
    if all(w == w[::-1] for w in group): bonus += 0.5 
    if len({w[0] for w in group}) == 1: bonus += 0.2
    return bonus


def find_best_partition(candidates, words_remaining):
    if not words_remaining:
        return [], 0.0

    best_partition = None
    best_score = -999.0

    for group, score in candidates:
        group_set = set(group)
        
        if group_set.issubset(words_remaining):
            remaining_after = words_remaining - group_set
            
            # Recurse
            sub_partition, sub_score = find_best_partition(candidates, remaining_after)
            
            if sub_partition is not None:
                current_total = score + sub_score
                if current_total > best_score:
                    best_score = current_total
                    best_partition = [group] + sub_partition
                    
    return best_partition, best_score

def get_best_move(words_list, model, embedder, banned_guesses):
    print("Global partitioning for best fit...")
    
    # Generate Candidates
    all_combos = list(itertools.combinations(words_list, 4))
    combo_inputs = [(", ".join(sorted(c)), "") for c in all_combos]
    base_scores = model.predict(combo_inputs)
    
    candidates = []
    for i, group in enumerate(all_combos):
        # Skip banned
        if tuple(sorted(group)) in banned_guesses:
            continue
            
        raw_score = base_scores[i]
        pat_bonus = check_patterns(group)
        coh_penalty = get_coherence_penalty(group, embedder)
        
        final_score = raw_score + pat_bonus + coh_penalty
        candidates.append((group, final_score))
        
    candidates.sort(key=lambda x: x[1], reverse=True)
    top_candidates = candidates[:SEARCH_DEPTH]
    
    # Find Global Partition
    best_partition, score = find_best_partition(top_candidates, set(words_list))
    
    if best_partition:
        # Return the group with the highest individual score in the partition
        best_group = max(best_partition, key=lambda g: [c[1] for c in candidates if c[0]==g][0])
        return best_group
    else:
        return top_candidates[0][0]

def play_smart():
    if not os.path.exists(MODEL_PATH):
        print("Error: Model not found.")
        return

    print("Loading Brains...")
    model = CrossEncoder(MODEL_PATH)
    embedder = SentenceTransformer('all-MiniLM-L6-v2') 
    
    remaining_words = {
        "may", "yank", "a", "card",
        "frozen", "produce", "dancing", "dairy",
        "make", "jay", "fast", "form",
        "drag", "firm", "mold", "tight"
    }
    
    lives = 4
    solved_count = 0
    banned_guesses = set() 

    print("\n--- GAME START ---")
    
    while solved_count < 4 and lives > 0:
        current_list = list(remaining_words)
        
        # GET BEST MOVE
        guess = get_best_move(current_list, model, embedder, banned_guesses)
        
        print(f"\n[Lives: {lives}] Best Guess: {guess}")
        
        res = input("Result? (c=Correct, o=One Away, w=Wrong): ").lower().strip()
        
        guess_tuple = tuple(sorted(guess))
        
        if res == 'c':
            print(" CORRECT! Removing words.")
            remaining_words -= set(guess)
            solved_count += 1
            # Clear outdated bans
            banned_guesses = {b for b in banned_guesses if set(b).issubset(remaining_words)}
            
        elif res == 'w':
            print(" WRONG. Re-calculating global strategy...")
            lives -= 1
            banned_guesses.add(guess_tuple)
            
        elif res == 'o':
            print(" ONE AWAY! Applying constraints...")
            lives -= 1
            banned_guesses.add(guess_tuple)
            candidates = apply_one_away_boost(candidates, guess)

    print("\n--- GAME OVER ---")
    if solved_count == 4: print("VICTORY!")
    else: print("DEFEAT.")

if __name__ == "__main__":
    play_smart()

Loading Brains...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- GAME START ---
   Thinking... (Scoring & Global Search)

[Lives: 4] Best Guess: ('fast', 'form', 'frozen', 'firm')
 >> ONE AWAY! Applying constraints...
   Thinking... (Scoring & Global Search)

[Lives: 3] Best Guess: ('mold', 'produce', 'dairy', 'a')
 >> WRONG. Re-calculating global strategy...
   Thinking... (Scoring & Global Search)

[Lives: 2] Best Guess: ('make', 'mold', 'produce', 'a')
 >> ONE AWAY! Applying constraints...
   Thinking... (Scoring & Global Search)

[Lives: 1] Best Guess: ('form', 'drag', 'a', 'card')
 >> WRONG. Re-calculating global strategy...

--- GAME OVER ---
DEFEAT.
